%md
# NB02 - Synthetic Pharmacy Claims
### Cross-Benefit Duplicate Drug Billing Detection - Pharmacy Leg

**Schema:** `cross_benefit_audit`<br>
**Tables created:** `member_registry`, `pharmacy_claims`, `seeded_duplicate_events`<br>
**Depends on:** NB01 - requires `cross_benefit_audit.target_hcpcs_crosswalk`

---

## What this notebook builds

Pharmacy claims arrive via **NCPDP D.0** transactions - the standard PBMs use to adjudicate retail and specialty pharmacy fills.<br>

Each claim carries an **NDC** (not a HCPCS code), a quantity, a days supply, and prescriber/pharmacy identifiers.

The bridge to the medical benefit is the **CMS ASP Crosswalk** from NB01.

| Table | Rows | Description |
|---|---|---|
| `member_registry` | 200 | Synthetic health plan members with PBM enrollment |
| `pharmacy_claims` | 552 | NCPDP-style specialty pharmacy fills, Oct–Dec 2026 |
| `seeded_duplicate_events` | 72 | Event registry linking seeded pharmacy fills to expected medical claims (NB03 reads this) |

### Seeded duplicate scenarios (72 of 552 claims)

| Scenario | Count | Description |
|---|---|---|
| `EXACT_SAME_CODE` | 20 | Same NDC→HCPCS on both legs; same or next-day service date |
| `NEAR_DATE` | 16 | Same NDC→HCPCS; pharmacy fill 1–3 days before medical claim |
| `QCODE_GAP` | 18 | Biosimilar Q-code NDC at pharmacy; originator J-code billed medically |
| `NEW_FORM_CROSS` | 10 | New-formulation SC NDC at pharmacy; originator IV code billed medically |
| `DIFF_BIOSIM` | 8 | One biosimilar NDC at pharmacy; different biosimilar Q-code billed medically |

## 0 - Setup

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
from datetime import date, timedelta
import uuid, random, hashlib, math

spark = SparkSession.builder.getOrCreate()

# Seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Tables
TARGET_TABLE = "cross_benefit_audit.target_hcpcs_crosswalk"
MEMBER_TABLE = "cross_benefit_audit.member_registry"
RX_TABLE     = "cross_benefit_audit.pharmacy_claims"
SEED_TABLE   = "cross_benefit_audit.seeded_duplicate_events"

PERIOD_START  = date(2026, 10, 1)
PERIOD_END    = date(2026, 12, 31)
N_MEMBERS     = 200
N_CLEAN       = 480

SCENARIO_COUNTS = {
    "EXACT_SAME_CODE": 20,
    "NEAR_DATE":       16,
    "QCODE_GAP":       18,
    "NEW_FORM_CROSS":  10,
    "DIFF_BIOSIM":      8,
}
N_SEEDED = sum(SCENARIO_COUNTS.values())   # 72 duplicate events

print("Setup complete.")
print(f"Period : {PERIOD_START} → {PERIOD_END}")
print(f"Members: {N_MEMBERS} | Clean fills: {N_CLEAN} | Seeded: {N_SEEDED}")
print(f"Total pharmacy claims: {N_CLEAN + N_SEEDED}")

Setup complete.
Period : 2026-10-01 → 2026-12-31
Members: 200 | Clean fills: 480 | Seeded: 72
Total pharmacy claims: 552


---
## 1 - Load target crosswalk

In [0]:
df_cross_walk = spark.table(TARGET_TABLE).toPandas()

print(f"Target crosswalk rows: {len(df_cross_walk)}")
print()
summary = (
    df_cross_walk.groupby(["molecule", "drug_type"])
    .agg(ndcs=("NDC", "count"), codes=("HCPCS_CODE", lambda x: " ".join(sorted(x.unique()))))
    .reset_index()
    .sort_values(["molecule", "drug_type"])
)
print(summary.to_string(index=False))

Target crosswalk rows: 31

     molecule       drug_type  ndcs             codes
  aflibercept      biosimilar     1             Q5138
  aflibercept      originator     2             J0178
  bevacizumab      biosimilar     6       Q5107 Q5118
  bevacizumab      originator     2             J9035
   infliximab      biosimilar     4 Q5103 Q5104 Q5121
   infliximab      originator     2             J1745
  ocrelizumab new_formulation     1             J2351
  ocrelizumab      originator     1             J2350
pembrolizumab new_formulation     2             J9277
pembrolizumab      originator     2             J9271
  ranibizumab      biosimilar     6       Q5124 Q5128
  ranibizumab      originator     2             J2778


---
## 2 - Drug-specific clinical parameters

Specialty drugs have well-defined dosing intervals, package sizes, and cost ranges. These drive realistic days-supply, quantity-dispensed, and ingredient-cost values in the synthetic claims.

In [0]:
# Drug clinical parameters by molecule
# price_per_fill: min_usd, max_usd; used for ingredient cost
# dosing_interval_days: most common interval; some drugs vary by weight/indication/other factors
# qty_units: billable units dispensed per fill (from BILLUNITSPKG in the crosswalk)
#            We will look up the actual crosswalk value; this is a fallback.
# daw_codes: weighted (code, weight) list for generating realistic DAW distributions
# DAW - dispense as written. 0 for generic, 1 and 7 for brand

DRUG_PARAMS = {
    "ranibizumab": {
        "dosing_interval_days": [28],         # monthly -intravitreal
        "days_supply_options": [28],
        "price_originator":   (1_900, 2_400),
        "price_biosimilar":   (1_100, 1_500),
        "daw_originator":     [(0, 0.4), (1, 0.6)],
        "daw_biosimilar":     [(0, 0.3), (7, 0.7)],
    },
    "aflibercept": {
        "dosing_interval_days": [28, 56],     # monthly or q8w
        "days_supply_options": [28, 56],
        "price_originator":   (2_100, 2_500),
        "price_biosimilar":   (1_500, 1_900),
        "daw_originator":     [(0, 0.4), (1, 0.6)],
        "daw_biosimilar":     [(0, 0.3), (7, 0.7)],
    },
    "bevacizumab": {
        "dosing_interval_days": [14, 21],     # q2-3w oncology infusion
        "days_supply_options": [14, 21],
        "price_originator":   (400, 700),
        "price_biosimilar":   (280, 450),
        "daw_originator":     [(0, 0.5), (1, 0.5)],
        "daw_biosimilar":     [(0, 0.4), (7, 0.6)],
    },
    "pembrolizumab": {
        "dosing_interval_days": [21, 42],     # q3w or q6w
        "days_supply_options": [21, 42],
        "price_originator":   (8_500, 11_000),
        "price_biosimilar":   (7_000, 9_000),  # Keytruda Qlex SC
        "daw_originator":     [(0, 0.5), (1, 0.5)],
        "daw_biosimilar":     [(0, 0.5), (7, 0.5)],
    },
    "infliximab": {
        "dosing_interval_days": [56],         # q8w maintenance; induction handled separately
        "days_supply_options": [56],
        "price_originator":   (2_500, 5_000), # 2-4 vials per infusion, weight-based
        "price_biosimilar":   (1_200, 3_000),
        "daw_originator":     [(0, 0.3), (1, 0.7)],
        "daw_biosimilar":     [(0, 0.4), (7, 0.6)],
    },
    "ocrelizumab": {
        "dosing_interval_days": [168],        # q6 months (two infusions per year)
        "days_supply_options": [168],
        "price_originator":   (16_000, 20_000),
        "price_biosimilar":   (14_000, 18_000),  # Ocrevus Zunovo SC
        "daw_originator":     [(0, 0.4), (1, 0.6)],
        "daw_biosimilar":     [(0, 0.4), (7, 0.6)],
    },
}

# Validate all target molecules covered
for mol in df_cross_walk["molecule"].unique():
    assert mol in DRUG_PARAMS, f"Missing drug params for: {mol}"

print("Drug parameters defined for:", sorted(DRUG_PARAMS.keys()))

Drug parameters defined for: ['aflibercept', 'bevacizumab', 'infliximab', 'ocrelizumab', 'pembrolizumab', 'ranibizumab']


---
## 3 - PBM and plan configuration

In NCPDP D.0, the pharmacy plan is identified by three fields:
- **BIN** (6-digit Bank Identification Number): routes the claim to the correct processor
- **PCN** (Processor Control Number): further routes within the PBM
- **Group**: the employer group or plan code

We define three fictional PBMs serving the synthetic 200 members.

After section 4, we also create a buy-and-bill provider pool.

In [0]:
# Fictional PBM configurations
PBM_PLANS = [
    # (pbm_name, bin, pcn, plan_type, weight)
    ("BlueRx",   "610591", "ADV",     "commercial",        0.40),
    ("RedPBM", "004336", "SPEC",    "medicare_advantage", 0.35),
    ("GreenRx",    "017832", "RXGRP",   "medicaid",          0.25),
]

# Fictional specialty pharmacy network
SPECIALTY_PHARMACIES = [
    # (name, state, npi_seed)
    ("A Specialty Pharmacy", "TN", 1001),
    ("B Specialty Pharmacy", "TX", 1002),
    ("C Specialty Pharmacy", "CA", 1003),
    ("D Specialty Pharmacy", "FL", 1004),
    ("E Specialty Pharmacy", "OH", 1005),
    ("F Specialty Pharmacy", "NY", 1006),
    ("G Specialty Pharmacy", "IL", 1007),
    ("H Specialty Pharmacy", "WA", 1008),
]

# US states for member geography
MEMBER_STATES = [
    "CA", "TX", "FL", "NY", "IL", "PA", "OH", "GA",
    "NC", "MI", "NJ", "AZ", "WA", "MA", "CO", "TN",
]

print(f"PBM plans      : {len(PBM_PLANS)}")
print(f"Pharmacies     : {len(SPECIALTY_PHARMACIES)}")
print(f"Member states  : {len(MEMBER_STATES)}")

PBM plans      : 3
Pharmacies     : 8
Member states  : 16


---
## 4 - Synthetic member population

We generate 200 health plan members. Each member is assigned:
- A PBM plan (drives BIN/PCN/group and plan type)
- A primary therapeutic area (drives which drug they'll be prescribed)
- A state of residence

In [0]:
def make_member_id(n: int) -> str: # a simple 6-digit incremental id generator preceded by 'MBR-'
    return f"MBR-{n:06d}"

def make_npi(seed_int: int) -> str:
    """10-digit NPI, starts with 1.""" 
    rng = random.Random(seed_int)
    digits = "".join(str(rng.randint(0, 9)) for i in range(9))
    return f"1{digits}"

def random_dob(plan_type: str, rng) -> date:
    """Age DOB by plan type."""
    if plan_type == "medicare_advantage":
        year = rng.randint(1940, 1961)   # 65-86 year-olds
    elif plan_type == "medicaid":
        year = rng.randint(1960, 2001)   # 25-66 year-olds
    else:
        year = rng.randint(1950, 1995)   # 31-76 year-olds (wider span)
    month = rng.randint(1, 12)
    day = rng.randint(1, 28)
    return date(year, month, day)

# Molecule assignment weighted by therapeutic area and plan type
MOLECULE_POOL = { # just the selected molecules
    "commercial":        ["infliximab", "bevacizumab", "pembrolizumab",
                          "ranibizumab", "aflibercept", "ocrelizumab"],
    "medicare_advantage": ["ranibizumab", "aflibercept", "bevacizumab",
                           "pembrolizumab", "infliximab", "ocrelizumab"],
    "medicaid":          ["infliximab", "ocrelizumab", "bevacizumab",
                          "pembrolizumab", "ranibizumab", "aflibercept"],
}

rng_members = random.Random(RANDOM_SEED + 1)
members = []
for i in range(1, N_MEMBERS + 1):
    pbm = rng_members.choices(PBM_PLANS, weights=[p[4] for p in PBM_PLANS])[0]
    pbm_name, bin_, pcn, plan_type, _ = pbm # bin_ to avoid Python's bin()
    group_id = f"GRP-{rng_members.randint(1000, 9999)}"
    mol_pool = MOLECULE_POOL[plan_type]
    molecule = rng_members.choice(mol_pool)
    state = rng_members.choice(MEMBER_STATES)
    dob = random_dob(plan_type, rng_members)
    members.append({
        "member_id":make_member_id(i),
        "dob": str(dob),
        "gender": rng_members.choice(["M", "F"]),
        "state": state,
        "plan_type": plan_type,
        "pbm_name": pbm_name,
        "pbm_bin": bin_,
        "pbm_pcn": pcn,
        "plan_group": group_id,
        "primary_molecule": molecule,
        "enrolled_date": str(date(2026, rng_members.randint(1, 9), 1)),
    })

df_members = pd.DataFrame(members)
print(f"Members generated: {len(df_members)}")
print()
print("Plan type distribution:")
print(df_members["plan_type"].value_counts().to_string())
print()
print("Primary molecule distribution:")
print(df_members["primary_molecule"].value_counts().to_string())

Members generated: 200

Plan type distribution:
plan_type
commercial            73
medicare_advantage    64
medicaid              63

Primary molecule distribution:
primary_molecule
bevacizumab      49
ranibizumab      33
ocrelizumab      31
pembrolizumab    30
infliximab       29
aflibercept      28


### Buy-and-bill provider pool.
These 5 providers appear as prescriber_npi on seeded pharmacy claims and as rendering_npi on the corresponding seeded medical claims (NB03 reads shared_prescriber_npi from seeded_duplicate_events and assigns the same NPI).

A deterministic **20%** of members are also flagged buy-and-bill eligible; their clean claims draw from this pool too, producing realistic FP NPI **co-location**. 

Seeds 9001-9005 are distinct from NB02 pharmacy NPI seeds (1001-1008) and NB03 provider seeds (2001-2033) and facility seeds (3001-3008).


In [0]:
BUY_AND_BILL_PROVIDERS = [
    {"name": "Dr. 001", "npi": make_npi(9001)},
    {"name": "Dr. 002", "npi": make_npi(9002)},
    {"name": "Dr. 003", "npi": make_npi(9003)},
    {"name": "Dr. 004", "npi": make_npi(9004)},
    {"name": "Dr. 005", "npi": make_npi(9005)},
]

def get_buy_and_bill_npi(member_id: str) -> str:
    """Deterministic buy-and-bill NPI for a member.
    Hashes member_id to one of the 5 buy-and-bill providers.
    NB03 uses the identical function so both notebooks assign the same NPI
    for the same member without sharing state."""
    idx = int(hashlib.md5(member_id.encode()).hexdigest(), 16) % len(BUY_AND_BILL_PROVIDERS)
    return BUY_AND_BILL_PROVIDERS[idx]["npi"]

def is_buy_and_bill_member(member_id: str) -> bool:
    """20% of members are buy-and-bill eligible (deterministic).
    NB03 uses the identical function for consistent NPI assignment across notebooks."""
    return int(hashlib.md5(member_id.encode()).hexdigest(), 16) % 5 == 0

print(f"Buy-and-bill providers : {len(BUY_AND_BILL_PROVIDERS)}")

Buy-and-bill providers : 5


---
## 5 - Helper functions for claim generation

In [0]:
def strip_ndc(ndc_formatted: str) -> str:
    """5-4-2 dashed NDC => 11-digit no-dash string."""
    return ndc_formatted.replace("-", "")

def random_date_in_period(rng, start: date = PERIOD_START, end: date = PERIOD_END) -> date:
    delta = (end - start).days
    return start + timedelta(days=rng.randint(0, delta))

def pick_ndc_for_molecule(df: pd.DataFrame, molecule: str, drug_type_filter=None, rng=None) -> pd.Series:
    """Return a random crosswalk row for the given molecule (filtered by drug_type)."""
    mask = df["molecule"] == molecule
    if drug_type_filter:
        mask &= df["drug_type"].isin(drug_type_filter if isinstance(drug_type_filter, list) else [drug_type_filter])
    pool = df[mask]
    if pool.empty:
        raise ValueError(f"No rows for molecule={molecule}, drug_type={drug_type_filter}")
    return pool.sample(1, random_state=rng.randint(0, 99999)).iloc[0]

def make_claim_id(service_date: date, seq: int) -> str:
    return f"RX-{service_date.strftime('%Y%m%d')}-{seq:05d}"

def make_seed_event_id(molecule: str, seq: int) -> str:
    prefix = molecule[:5].upper()
    return f"EVT-{prefix}-{seq:03d}"

def choose_daw(drug_type: str, params: dict, rng) -> int: # DAW = dispense as written
    key = "daw_originator" if drug_type in ("originator",) else "daw_biosimilar"
    codes, weights = zip(*params[key])
    return rng.choices(codes, weights=weights)[0]

def ingredient_cost(drug_type: str, params: dict, rng) -> float:
    lo, hi = params["price_originator"] if drug_type == "originator" else params["price_biosimilar"]
    return round(rng.uniform(lo, hi), 2)

def build_claim(seq: int, member_row: pd.Series, cross_walk_row: pd.Series,
                service_date: date, rng,
                is_seeded: bool = False, seed_event_id: str = None,
                seed_scenario: str = None,
                override_prescriber_npi: str = None,
                override_qty_dispensed: float = None) -> dict:
    """Assemble one pharmacy claim record.

    override_prescriber_npi: when provided, bypasses the per-claim random NPI and uses
        the shared buy-and-bill NPI from the seeded event (coordinated with NB03).
        For buy-and-bill-eligible clean claims, the override is not passed here; the
        eligibility check below handles those instead.
    override_qty_dispensed: when provided, replaces the default BILLUNITSPKG quantity
        with the dose-first derived value (administered_hcpcs_units * BILLUNITS * noise).
        Only passed for seeded claims; clean claims retain BILLUNITSPKG."""
    molecule  = cross_walk_row["molecule"]
    drug_type = cross_walk_row["drug_type"]
    params    = DRUG_PARAMS[molecule]

    days_supply = rng.choice(params["days_supply_options"])
    daw         = choose_daw(drug_type, params, rng)
    cost        = ingredient_cost(drug_type, params, rng)
    disp_fee    = round(rng.uniform(2.50, 12.00), 2)
    copay       = round(rng.uniform(0.0, 150.0), 2) if member_row["plan_type"] != "medicaid" else 0.0
    plan_paid   = round(cost + disp_fee - copay, 2)

    pharm = rng.choice(SPECIALTY_PHARMACIES)
    ph_name, ph_state, ph_npi_seed = pharm
    ph_npi = make_npi(ph_npi_seed * 10000 + seq)

    # Prescriber NPI: seeded claims use the shared buy-and-bill NPI (coordinated with NB03);
    # buy-and-bill-eligible members' clean claims draw from the same pool;
    # all other clean claims keep the original per-claim hash (no rng call).
    if override_prescriber_npi is not None:
        px_npi = override_prescriber_npi
    elif is_buy_and_bill_member(member_row["member_id"]):
        px_npi = get_buy_and_bill_npi(member_row["member_id"])
    else:
        px_npi = make_npi(RANDOM_SEED * 100000 + seq * 7 + 3)

    # quantity_dispensed: seeded claims use the dose-first derived value so that
    # concordance_ratio = (qty / median_billunits) / med_units ≈ 1.0 for TPs;
    # clean claims retain BILLUNITSPKG (original behavior, produces low FP ratio).
    if override_qty_dispensed is not None:
        qty = override_qty_dispensed
    else:
        qty = float(cross_walk_row["BILLUNITSPKG"])

    return {
        "claim_id":          make_claim_id(service_date, seq),
        "member_id":         member_row["member_id"],
        "service_date":      str(service_date),
        "paid_date":         str(service_date + timedelta(days=rng.randint(1, 5))),
        "submitted_ndc":     strip_ndc(cross_walk_row["NDC"]),
        "ndc_formatted":     cross_walk_row["NDC"],
        "quantity_dispensed": qty,
        "days_supply":       days_supply,
        "refill_number":     rng.choice([0, 0, 0, 1, 1, 2, 3]),
        "daw_code":          daw,
        "prescriber_npi":    px_npi,
        "pharmacy_npi":      ph_npi,
        "pharmacy_name":     ph_name,
        "pharmacy_state":    ph_state,
        "pbm_bin":           member_row["pbm_bin"],
        "pbm_pcn":           member_row["pbm_pcn"],
        "plan_group":        member_row["plan_group"],
        "ingredient_cost":   cost,
        "dispensing_fee":    disp_fee,
        "member_copay":      copay,
        "plan_paid":         plan_paid,
        "total_cost":        round(cost + disp_fee, 2),
        # crosswalk-derived
        "hcpcs_code":        cross_walk_row["HCPCS_CODE"],
        "molecule":          molecule,
        "brand_family":      cross_walk_row["brand_family"],
        "drug_type":         drug_type,
        "therapeutic_area":  cross_walk_row["therapeutic_area"],
        # duplicate detection flags
        "is_seeded_duplicate": is_seeded,
        "seed_event_id":     seed_event_id,
        "seed_scenario_type": seed_scenario,
    }

print("Helper functions defined.")

Helper functions defined.


![logic_flow_claim_building.png](./logic_flow_claim_building.png "logic_flow_claim_building.png")

---
## 6 - Generate clean pharmacy claims (480 fills)

These 480 fills have no corresponding medical claim counterpart.<br>
They represent legitimate specialty pharmacy fills where the drug was dispensed but **not** billed separately through the medical benefit.

In [0]:
rng_clean  = random.Random(RANDOM_SEED + 10)
clean_rows = []
claim_seq  = 1

for i in range(N_CLEAN):
    mbr = df_members.sample(1, random_state=rng_clean.randint(0, 999999)).iloc[0]
    molecule = mbr["primary_molecule"]

    # Clean claims mix. Mostly originators and some biosimilars
    drug_type_filter = rng_clean.choices(
        ["originator", "biosimilar", "new_formulation"],
        weights=[0.55, 0.35, 0.10],
    )[0]
    # Not all molecules have biosimilars/new_formulations in scope
    mol_df = df_cross_walk[df_cross_walk["molecule"] == molecule]
    avail_types = mol_df["drug_type"].unique().tolist()
    if drug_type_filter not in avail_types:
        drug_type_filter = rng_clean.choice(avail_types)

    cross_walk_row   = pick_ndc_for_molecule(df_cross_walk, molecule, drug_type_filter, rng_clean)
    svc_date    = random_date_in_period(rng_clean)

    row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_clean,
        is_seeded=False,
    )
    clean_rows.append(row)
    claim_seq += 1

df_clean = pd.DataFrame(clean_rows)
print(f"Clean claims generated: {len(df_clean)}")
print()
print("Molecule distribution:")
print(df_clean["molecule"].value_counts().to_string())
print()
print("Drug type distribution:")
print(df_clean["drug_type"].value_counts().to_string())

Clean claims generated: 480

Molecule distribution:
molecule
bevacizumab      113
pembrolizumab     80
ocrelizumab       75
infliximab        72
ranibizumab       71
aflibercept       69

Drug type distribution:
drug_type
originator         319
biosimilar         121
new_formulation     40


---
## 7 - Generate seeded duplicate scenarios (72 fills)

Each seeded claim becomes one entry in `seeded_duplicate_events`. The notebook NB03 reads it to generate the corresponding medical benefit claim.

In [0]:
# Per-molecule originator HCPCS lookup
# Maps molecule ==> the J-code that infusion centers bill for the originator drug
ORIGINATOR_HCPCS = {
    row["molecule"]: row["HCPCS_CODE"]
    for _, row in df_cross_walk[df_cross_walk["drug_type"] == "originator"].iterrows()
}
# Some molecules only have one originator code but let's resolve duplicates
# infliximab/bevacizumab/ranibizumab each have one J-code
print("Originator HCPCS map:")
for mol, code in sorted(ORIGINATOR_HCPCS.items()):
    print(f"  {mol:<18} --> {code}")

# Units range per expected medical HCPCS code -- mirrors HCPCS_PARAMS.units_range in NB03.
# Used by the seeded scenario loops to derive administered_hcpcs_units for the
# dose-first concordance derivation: qty_dispensed = adm_units * BILLUNITS * noise,
# which yields concordance_ratio = (qty / median_billunits) / med_units ≈ 1.0 for TPs.
# Changes here must be reflected in NB03's HCPCS_PARAMS.
MED_HCPCS_UNITS_RANGE = {
    "J2778": (3, 5),    # ranibizumab originator (0.1 mg/unit; 0.3--0.5 mg dose)
    "Q5124": (5, 5),    # ranibizumab biosimilar
    "Q5128": (3, 5),    # ranibizumab biosimilar
    "J0178": (2, 2),    # aflibercept originator (1 mg/unit; 2 mg standard dose)
    "Q5138": (2, 2),    # aflibercept biosimilar
    "J9035": (40, 70),  # bevacizumab originator (10 mg/unit; 400--700 mg oncology dose)
    "Q5107": (40, 70),  # bevacizumab biosimilar
    "Q5118": (40, 70),  # bevacizumab biosimilar
    "J9271": (200, 200),# pembrolizumab originator (1 mg/unit; 200 mg flat dose)
    "J9277": (400, 400),# pembrolizumab new formulation SC (1 mg/unit; 400 mg dose)
    "J1745": (30, 45),  # infliximab originator (10 mg/unit; 5 mg/kg weight-based)
    "Q5103": (30, 45),  # infliximab biosimilar
    "Q5104": (30, 45),  # infliximab biosimilar
    "Q5121": (30, 45),  # infliximab biosimilar
    "J2350": (600, 600),# ocrelizumab IV (1 mg/unit; 600 mg standard dose)
    "J2351": (920, 920),# ocrelizumab SC new formulation (1 mg/unit; 920 mg dose)
}

Originator HCPCS map:
  aflibercept        --> J0178
  bevacizumab        --> J9035
  infliximab         --> J1745
  ocrelizumab        --> J2350
  pembrolizumab      --> J9271
  ranibizumab        --> J2778


In [0]:
rng_seed      = random.Random(RANDOM_SEED + 20)
seeded_claims = []
seed_events   = []
seed_seq      = 1

# Separate RNG for administered dose derivation.
# Using a distinct RNG instance (not rng_seed) preserves the rng_seed sequence
# so that member/drug/date selections in each scenario loop are identical to
# the pre-fix version -- only quantity_dispensed and prescriber_npi change.
rng_adm = random.Random(RANDOM_SEED + 500)

# Reserves members for seeded scenarios
# We draw from the bottom 100 member IDs to avoid overlap with clean claims'
# heavy rotation of the top members.
seed_member_pool = df_members[df_members["member_id"] >= "MBR-000101"].copy()

def pick_seed_member(mol: str, rng) -> pd.Series:
    """Pick a member whose primary molecule matches (or fall back to any)."""
    pool = seed_member_pool[seed_member_pool["primary_molecule"] == mol]
    if len(pool) == 0:
        pool = seed_member_pool
    return pool.sample(1, random_state=rng.randint(0, 999999)).iloc[0]


# 7.1 EXACT_SAME_CODE (20 scenarios)
# Pharmacy fill, any target NDC --> medical claim: same HCPCS code, same service date
# Most straightforward duplicate - same drug, same date, both benefits billed

for i in range(SCENARIO_COUNTS["EXACT_SAME_CODE"]):
    mol = rng_seed.choice(sorted(df_cross_walk["molecule"].unique()))
    mbr = pick_seed_member(mol, rng_seed)
    cross_walk_row = pick_ndc_for_molecule(df_cross_walk, mol, rng=rng_seed)
    svc_date = random_date_in_period(rng_seed)
    eid = make_seed_event_id(mol, seed_seq)

    # Dose-first derivation: draw administered units from the medical HCPCS code range;
    # back-calculate pharmacy quantity = adm_units * BILLUNITS * noise so that
    # concordance_ratio = (qty / median_billunits) / med_units ≈ noise_ratio ≈ 1.0.
    # rng_adm is separate from rng_seed; no disruption to mol/member/date sequence.
    units_lo, units_hi = MED_HCPCS_UNITS_RANGE[cross_walk_row["HCPCS_CODE"]]
    adm_units   = rng_adm.randint(units_lo, units_hi)
    qty_noise   = rng_adm.uniform(0.92, 1.08)
    override_qty = round(adm_units * float(cross_walk_row["BILLUNITS"]) * qty_noise, 2)
    # NPI: cycle through buy-and-bill pool by seed_seq (deterministic, no rng call)
    shared_npi  = BUY_AND_BILL_PROVIDERS[seed_seq % len(BUY_AND_BILL_PROVIDERS)]["npi"]

    rx_row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_seed,
        is_seeded=True, seed_event_id=eid, seed_scenario="EXACT_SAME_CODE",
        override_prescriber_npi=shared_npi,
        override_qty_dispensed=override_qty,
    )
    seeded_claims.append(rx_row)
    seed_events.append({
        "seed_event_id":          eid,
        "scenario_type":          "EXACT_SAME_CODE",
        "member_id":              mbr["member_id"],
        "molecule":               mol,
        "rx_claim_id":            rx_row["claim_id"],
        "rx_ndc":                 rx_row["submitted_ndc"],
        "rx_hcpcs_code":          cross_walk_row["HCPCS_CODE"],
        "rx_service_date":        str(svc_date),
        "expected_med_hcpcs":     cross_walk_row["HCPCS_CODE"],   # same code on medical side
        "med_date_offset":        0,                           # same service date
        "shared_prescriber_npi":  shared_npi,
        "administered_hcpcs_units": float(adm_units),
        "description": f"Exact duplicate: {cross_walk_row['brand_family']} dispensed at pharmacy AND billed on medical claim same day",
    })
    claim_seq += 1
    seed_seq  += 1

print(f"EXACT_SAME_CODE scenarios generated: {SCENARIO_COUNTS['EXACT_SAME_CODE']}")

EXACT_SAME_CODE scenarios generated: 20


In [0]:
# 7.2 NEAR_DATE (16 scenarios)
# Pharmacy fill 1--3 days before the infusion center claim
# Represents patient fills drug at specialty pharmacy, brings it to infusion
# center the next day or two later --> infusion center also bills the drug

for i in range(SCENARIO_COUNTS["NEAR_DATE"]):
    mol = rng_seed.choice(sorted(df_cross_walk["molecule"].unique()))
    mbr = pick_seed_member(mol, rng_seed)
    cross_walk_row = pick_ndc_for_molecule(df_cross_walk, mol, rng=rng_seed)
    svc_date = random_date_in_period(rng_seed, end=date(2026, 12, 28))
    offset = rng_seed.randint(1, 3)
    eid = make_seed_event_id(mol, seed_seq)

    # Dose-first derivation (same pattern as EXACT_SAME_CODE).
    units_lo, units_hi = MED_HCPCS_UNITS_RANGE[cross_walk_row["HCPCS_CODE"]]
    adm_units   = rng_adm.randint(units_lo, units_hi)
    qty_noise   = rng_adm.uniform(0.92, 1.08)
    override_qty = round(adm_units * float(cross_walk_row["BILLUNITS"]) * qty_noise, 2)
    shared_npi  = BUY_AND_BILL_PROVIDERS[seed_seq % len(BUY_AND_BILL_PROVIDERS)]["npi"]

    rx_row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_seed,
        is_seeded=True, seed_event_id=eid, seed_scenario="NEAR_DATE",
        override_prescriber_npi=shared_npi,
        override_qty_dispensed=override_qty,
    )
    seeded_claims.append(rx_row)
    seed_events.append({
        "seed_event_id":          eid,
        "scenario_type":          "NEAR_DATE",
        "member_id":              mbr["member_id"],
        "molecule":               mol,
        "rx_claim_id":            rx_row["claim_id"],
        "rx_ndc":                 rx_row["submitted_ndc"],
        "rx_hcpcs_code":          cross_walk_row["HCPCS_CODE"],
        "rx_service_date":        str(svc_date),
        "expected_med_hcpcs":     cross_walk_row["HCPCS_CODE"],
        "med_date_offset":        offset,
        "shared_prescriber_npi":  shared_npi,
        "administered_hcpcs_units": float(adm_units),
        "description": f"Near-date duplicate: pharmacy fill {offset}d before infusion center bills {cross_walk_row['HCPCS_CODE']}",
    })
    claim_seq += 1
    seed_seq  += 1

print(f"NEAR_DATE scenarios generated: {SCENARIO_COUNTS['NEAR_DATE']}")

NEAR_DATE scenarios generated: 16


In [0]:
# 7.3 QCODE_GAP (18 scenarios)
# Pharmacy fills a BIOSIMILAR (Q-code NDC) --> infusion center bills ORIGINATOR J-code
# This is the hardest scenario for naive detection engines:
#   - NDC maps to Q5103 (Inflectra)
#   - Medical claim is J1745 (Remicade originator)
#   - A J-to-J code matcher misses this entirely
#
# Molecules with biosimilar Q-codes: infliximab, bevacizumab, ranibizumab, aflibercept

QCODE_GAP_MOLECULES = [m for m in df_cross_walk["molecule"].unique()
                       if "biosimilar" in df_cross_walk[df_cross_walk["molecule"] == m]["drug_type"].values]

for i in range(SCENARIO_COUNTS["QCODE_GAP"]):
    mol = rng_seed.choice(QCODE_GAP_MOLECULES)
    mbr = pick_seed_member(mol, rng_seed)
    cross_walk_row = pick_ndc_for_molecule(df_cross_walk, mol, drug_type_filter="biosimilar", rng=rng_seed)
    svc_date = random_date_in_period(rng_seed)
    eid = make_seed_event_id(mol, seed_seq)
    orig_code = ORIGINATOR_HCPCS[mol]    # the J-code infusion center bills

    # Dose-first derivation: administered units come from the MEDICAL (J-code) range.
    # The pharmacy fills a biosimilar Q-code but the clinical dose is the same molecule;
    # both biosimilar and originator bill the same mg/unit so BILLUNITS is equivalent.
    units_lo, units_hi = MED_HCPCS_UNITS_RANGE[orig_code]
    adm_units   = rng_adm.randint(units_lo, units_hi)
    qty_noise   = rng_adm.uniform(0.92, 1.08)
    # qty formula uses pharmacy HCPCS (Q-code) BILLUNITS -- NB05 groups median_billunits
    # by rx_hcpcs, so the concordance denominator is the Q-code's BILLUNITS.
    override_qty = round(adm_units * float(cross_walk_row["BILLUNITS"]) * qty_noise, 2)
    shared_npi  = BUY_AND_BILL_PROVIDERS[seed_seq % len(BUY_AND_BILL_PROVIDERS)]["npi"]

    rx_row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_seed,
        is_seeded=True, seed_event_id=eid, seed_scenario="QCODE_GAP",
        override_prescriber_npi=shared_npi,
        override_qty_dispensed=override_qty,
    )
    seeded_claims.append(rx_row)
    seed_events.append({
        "seed_event_id":          eid,
        "scenario_type":          "QCODE_GAP",
        "member_id":              mbr["member_id"],
        "molecule":               mol,
        "rx_claim_id":            rx_row["claim_id"],
        "rx_ndc":                 rx_row["submitted_ndc"],
        "rx_hcpcs_code":          cross_walk_row["HCPCS_CODE"],   # Q-code
        "rx_service_date":        str(svc_date),
        "expected_med_hcpcs":     orig_code,                  # originator J-code - different!
        "med_date_offset":        rng_seed.randint(0, 1),
        "shared_prescriber_npi":  shared_npi,
        "administered_hcpcs_units": float(adm_units),
        "description": (
            f"Q-code gap: {cross_walk_row['brand_family']} ({cross_walk_row['HCPCS_CODE']}) "
            f"dispensed at pharmacy; infusion center bills originator {orig_code}. "
            f"Naive J-to-J matcher misses this."
        ),
    })
    claim_seq += 1
    seed_seq  += 1

print(f"QCODE_GAP scenarios generated: {SCENARIO_COUNTS['QCODE_GAP']}")

QCODE_GAP scenarios generated: 18


In [0]:
# 7.4 NEW_FORM_CROSS (10 scenarios)
# Patient fills NEW FORMULATION (SC) at specialty pharmacy
# and the infusion center bills the ORIGINAL IV formulation J-code
#
# Pairs:
#   J2351 (Ocrevus Zunovo SC) <--> J2350 (Ocrevus IV)
#   J9277 (Keytruda Qlex SC)  <--> J9271 (Keytruda IV)

NEW_FORM_PAIRS = {
    "J2351": "J2350",   # ocrelizumab SC --> IV
    "J9277": "J9271",   # pembrolizumab SC --> IV
}

NEW_FORM_SC_CODES = list(NEW_FORM_PAIRS.keys())

for i in range(SCENARIO_COUNTS["NEW_FORM_CROSS"]):
    sc_code = rng_seed.choice(NEW_FORM_SC_CODES)
    mol = df_cross_walk[df_cross_walk["HCPCS_CODE"] == sc_code]["molecule"].iloc[0]
    mbr = pick_seed_member(mol, rng_seed)
    cross_walk_row = pick_ndc_for_molecule(df_cross_walk, mol, drug_type_filter="new_formulation", rng=rng_seed)
    svc_date  = random_date_in_period(rng_seed)
    eid = make_seed_event_id(mol, seed_seq)
    med_code = NEW_FORM_PAIRS[cross_walk_row["HCPCS_CODE"]]

    # Dose-first derivation: administered units from the MEDICAL (IV J-code) range.
    # The infusion center bills the IV code; NB03 will record med_units from that range.
    units_lo, units_hi = MED_HCPCS_UNITS_RANGE[med_code]
    adm_units   = rng_adm.randint(units_lo, units_hi)
    qty_noise   = rng_adm.uniform(0.92, 1.08)
    # qty formula uses pharmacy HCPCS (SC code) BILLUNITS for denominator consistency.
    override_qty = round(adm_units * float(cross_walk_row["BILLUNITS"]) * qty_noise, 2)
    shared_npi  = BUY_AND_BILL_PROVIDERS[seed_seq % len(BUY_AND_BILL_PROVIDERS)]["npi"]

    rx_row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_seed,
        is_seeded=True, seed_event_id=eid, seed_scenario="NEW_FORM_CROSS",
        override_prescriber_npi=shared_npi,
        override_qty_dispensed=override_qty,
    )
    seeded_claims.append(rx_row)
    seed_events.append({
        "seed_event_id":          eid,
        "scenario_type":          "NEW_FORM_CROSS",
        "member_id":              mbr["member_id"],
        "molecule":               mol,
        "rx_claim_id":            rx_row["claim_id"],
        "rx_ndc":                 rx_row["submitted_ndc"],
        "rx_hcpcs_code":          cross_walk_row["HCPCS_CODE"],
        "rx_service_date":        str(svc_date),
        "expected_med_hcpcs":     med_code,
        "med_date_offset":        rng_seed.randint(0, 2),
        "shared_prescriber_npi":  shared_npi,
        "administered_hcpcs_units": float(adm_units),
        "description": (
            f"Formulation cross: {cross_walk_row['brand_family']} ({cross_walk_row['HCPCS_CODE']} SC) "
            f"filled at pharmacy; infusion center bills IV equivalent {med_code}."
        ),
    })
    claim_seq += 1
    seed_seq  += 1

print(f"NEW_FORM_CROSS scenarios generated: {SCENARIO_COUNTS['NEW_FORM_CROSS']}")

NEW_FORM_CROSS scenarios generated: 10


In [0]:
# 7.5 DIFF_BIOSIM (8 scenarios)
# Patient fills ONE biosimilar at pharmacy and theinfusion center bills
# a DIFFERENT biosimilar for the same molecule
# Example: specialty pharmacy dispenses Renflexis (Q5104)
#          infusion center bills Inflectra (Q5103)
#
# Only works for molecules with 2+ biosimilar Q-codes: infliximab, bevacizumab, ranibizumab

DIFF_BIOSIM_MOLS = {
    mol: df_cross_walk[(df_cross_walk["molecule"] == mol) & (df_cross_walk["drug_type"] == "biosimilar")]["HCPCS_CODE"].unique().tolist()
    for mol in df_cross_walk["molecule"].unique()
}
DIFF_BIOSIM_MOLS = {mol: codes for mol, codes in DIFF_BIOSIM_MOLS.items() if len(codes) >= 2}
print("Molecules with 2+ biosimilar Q-codes:", list(DIFF_BIOSIM_MOLS.keys()))

for i in range(SCENARIO_COUNTS["DIFF_BIOSIM"]):
    mol = rng_seed.choice(list(DIFF_BIOSIM_MOLS.keys()))
    mbr = pick_seed_member(mol, rng_seed)

    # Pick two distinct biosimilar HCPCS codes for this molecule
    codes     = DIFF_BIOSIM_MOLS[mol]
    rx_code, med_code = rng_seed.sample(codes, 2)

    # Pharmacy fills the first biosimilar
    rx_pool = df_cross_walk[(df_cross_walk["molecule"] == mol) & (df_cross_walk["HCPCS_CODE"] == rx_code)]
    cross_walk_row = rx_pool.sample(1, random_state=rng_seed.randint(0, 999999)).iloc[0]

    svc_date = random_date_in_period(rng_seed)
    eid = make_seed_event_id(mol, seed_seq)

    # Dose-first derivation: both biosimilars share the same clinical dose (same molecule);
    # draw from the medical biosimilar's range (equivalent to the pharmacy biosimilar's range).
    units_lo, units_hi = MED_HCPCS_UNITS_RANGE[med_code]
    adm_units   = rng_adm.randint(units_lo, units_hi)
    qty_noise   = rng_adm.uniform(0.92, 1.08)
    override_qty = round(adm_units * float(cross_walk_row["BILLUNITS"]) * qty_noise, 2)
    shared_npi  = BUY_AND_BILL_PROVIDERS[seed_seq % len(BUY_AND_BILL_PROVIDERS)]["npi"]

    rx_row = build_claim(
        seq=claim_seq, member_row=mbr, cross_walk_row=cross_walk_row,
        service_date=svc_date, rng=rng_seed,
        is_seeded=True, seed_event_id=eid, seed_scenario="DIFF_BIOSIM",
        override_prescriber_npi=shared_npi,
        override_qty_dispensed=override_qty,
    )
    seeded_claims.append(rx_row)
    seed_events.append({
        "seed_event_id":          eid,
        "scenario_type":          "DIFF_BIOSIM",
        "member_id":              mbr["member_id"],
        "molecule":               mol,
        "rx_claim_id":            rx_row["claim_id"],
        "rx_ndc":                 rx_row["submitted_ndc"],
        "rx_hcpcs_code":          rx_code,
        "rx_service_date":        str(svc_date),
        "expected_med_hcpcs":     med_code,
        "med_date_offset":        0,
        "shared_prescriber_npi":  shared_npi,
        "administered_hcpcs_units": float(adm_units),
        "description": (
            f"Biosimilar substitution: {rx_code} at pharmacy; "
            f"infusion center bills different biosimilar {med_code} for same molecule ({mol})."
        ),
    })
    claim_seq += 1
    seed_seq  += 1

print(f"DIFF_BIOSIM scenarios generated: {SCENARIO_COUNTS['DIFF_BIOSIM']}")

Molecules with 2+ biosimilar Q-codes: ['infliximab', 'ranibizumab', 'bevacizumab']
DIFF_BIOSIM scenarios generated: 8


---
## 8 - Combine and finalize claim dataset

In [0]:
df_seeded = pd.DataFrame(seeded_claims)
df_claims = pd.concat([df_clean, df_seeded], ignore_index=True)

# Ensure correct datetime and bool types
df_claims["service_date"]  = pd.to_datetime(df_claims["service_date"]).dt.date
df_claims["paid_date"]     = pd.to_datetime(df_claims["paid_date"]).dt.date
df_claims["is_seeded_duplicate"] = df_claims["is_seeded_duplicate"].astype(bool)

# Column order
CLAIM_COLS = [
    "claim_id", "member_id", "service_date", "paid_date",
    "submitted_ndc", "ndc_formatted", "quantity_dispensed", "days_supply",
    "refill_number", "daw_code",
    "prescriber_npi", "pharmacy_npi", "pharmacy_name", "pharmacy_state",
    "pbm_bin", "pbm_pcn", "plan_group",
    "ingredient_cost", "dispensing_fee", "member_copay", "plan_paid", "total_cost",
    "hcpcs_code", "molecule", "brand_family", "drug_type", "therapeutic_area",
    "is_seeded_duplicate", "seed_event_id", "seed_scenario_type",
]
df_claims = df_claims[CLAIM_COLS]

df_seed_events = pd.DataFrame(seed_events)
df_seed_events["rx_service_date"] = pd.to_datetime(df_seed_events["rx_service_date"]).dt.date

print(f"Total pharmacy claims : {len(df_claims):,}")
print(f"  Clean               : {(~df_claims['is_seeded_duplicate']).sum():,}")
print(f"  Seeded              : {df_claims['is_seeded_duplicate'].sum():,}")
print()
print("Seeded scenario breakdown:")
print(df_claims[df_claims["is_seeded_duplicate"]]["seed_scenario_type"].value_counts().to_string())
print()
print("Total seeded events   :", len(df_seed_events))
print()
print("seeded_duplicate_events columns:", list(df_seed_events.columns))

Total pharmacy claims : 552
  Clean               : 480
  Seeded              : 72

Seeded scenario breakdown:
seed_scenario_type
EXACT_SAME_CODE    20
QCODE_GAP          18
NEAR_DATE          16
NEW_FORM_CROSS     10
DIFF_BIOSIM         8

Total seeded events   : 72

seeded_duplicate_events columns: ['seed_event_id', 'scenario_type', 'member_id', 'molecule', 'rx_claim_id', 'rx_ndc', 'rx_hcpcs_code', 'rx_service_date', 'expected_med_hcpcs', 'med_date_offset', 'shared_prescriber_npi', 'administered_hcpcs_units', 'description']


---
## 9 - Write to Delta Lake

In [0]:
# 9a - member_registry

sdf_members = spark.createDataFrame(df_members)

(
    sdf_members.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MEMBER_TABLE)
)

print(f"Written: {MEMBER_TABLE}")
print(f"Row count: {spark.table(MEMBER_TABLE).count():,}")

Written: cross_benefit_audit.member_registry
Row count: 200


In [0]:
# 9b - pharmacy_claims

sdf_claims = spark.createDataFrame(df_claims)

(
    sdf_claims.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RX_TABLE)
)

print(f"Written: {RX_TABLE}")
print(f"Row count: {spark.table(RX_TABLE).count():,}")

Written: cross_benefit_audit.pharmacy_claims
Row count: 552


In [0]:
# 9c - seeded_duplicate_events

sdf_seed = spark.createDataFrame(df_seed_events)

(
    sdf_seed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SEED_TABLE)
)

print(f"Written: {SEED_TABLE}")
print(f"Row count: {spark.table(SEED_TABLE).count():,}")

Written: cross_benefit_audit.seeded_duplicate_events
Row count: 72


---
## 10 - Sanity checks after table write

In [0]:
# Schema review
print(f"=== {RX_TABLE} schema ===")
spark.table(RX_TABLE).printSchema()

=== cross_benefit_audit.pharmacy_claims schema ===
root
 |-- claim_id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- service_date: date (nullable = true)
 |-- paid_date: date (nullable = true)
 |-- submitted_ndc: string (nullable = true)
 |-- ndc_formatted: string (nullable = true)
 |-- quantity_dispensed: double (nullable = true)
 |-- days_supply: long (nullable = true)
 |-- refill_number: long (nullable = true)
 |-- daw_code: long (nullable = true)
 |-- prescriber_npi: string (nullable = true)
 |-- pharmacy_npi: string (nullable = true)
 |-- pharmacy_name: string (nullable = true)
 |-- pharmacy_state: string (nullable = true)
 |-- pbm_bin: string (nullable = true)
 |-- pbm_pcn: string (nullable = true)
 |-- plan_group: string (nullable = true)
 |-- ingredient_cost: double (nullable = true)
 |-- dispensing_fee: double (nullable = true)
 |-- member_copay: double (nullable = true)
 |-- plan_paid: double (nullable = true)
 |-- total_cost: double (nullable = true)

In [0]:
# All seeded events have a corresponding claim
seed_claim_ids  = set(df_seed_events["rx_claim_id"])
claims_in_table = set(spark.table(RX_TABLE).filter("is_seeded_duplicate = true").select("claim_id").toPandas()["claim_id"])
orphan_events   = seed_claim_ids - claims_in_table

print(f"Seeded events registered    : {len(df_seed_events)}")
print(f"Seeded claims in RX table   : {len(claims_in_table)}")
print(f"Orphaned seed events        : {len(orphan_events)}")  # should be 0

Seeded events registered    : 72
Seeded claims in RX table   : 72
Orphaned seed events        : 0


In [0]:
# QCODE_GAP - show the pharmacy code vs expected medical code
print("QCODE_GAP scenario sample:")
qgap = df_seed_events[df_seed_events["scenario_type"] == "QCODE_GAP"][
    ["molecule", "rx_hcpcs_code", "expected_med_hcpcs", "description"]
].head(6)
for _, r in qgap.iterrows():
    print(f"  {r['molecule']:<18} pharmacy:{r['rx_hcpcs_code']}  medical:{r['expected_med_hcpcs']}")

QCODE_GAP scenario sample:
  ranibizumab        pharmacy:Q5128  medical:J2778
  ranibizumab        pharmacy:Q5128  medical:J2778
  aflibercept        pharmacy:Q5138  medical:J0178
  bevacizumab        pharmacy:Q5107  medical:J9035
  bevacizumab        pharmacy:Q5118  medical:J9035
  infliximab         pharmacy:Q5121  medical:J1745


In [0]:
# Cost distribution by molecule
print("Average ingredient cost per fill by molecule and drug type:")
cost_summary = (
    spark.table(RX_TABLE)
    .groupBy("molecule", "drug_type")
    .agg(
        {"ingredient_cost": "avg", "claim_id": "count"}
    )
    .withColumnRenamed("avg(ingredient_cost)", "avg_cost")
    .withColumnRenamed("count(claim_id)", "n_fills")
    .orderBy("molecule", "drug_type")
)
cost_summary.show(20, truncate=False)

Average ingredient cost per fill by molecule and drug type:
+-------------+---------------+------------------+-------+
|molecule     |drug_type      |avg_cost          |n_fills|
+-------------+---------------+------------------+-------+
|aflibercept  |biosimilar     |1727.80375        |24     |
|aflibercept  |originator     |2292.995          |54     |
|bevacizumab  |biosimilar     |367.82807692307694|52     |
|bevacizumab  |originator     |549.0606944444445 |72     |
|infliximab   |biosimilar     |1979.1762162162167|37     |
|infliximab   |originator     |3941.7500000000005|48     |
|ocrelizumab  |new_formulation|15818.970384615384|26     |
|ocrelizumab  |originator     |18131.998928571433|56     |
|pembrolizumab|new_formulation|8064.447096774195 |31     |
|pembrolizumab|originator     |9687.542835820897 |67     |
|ranibizumab  |biosimilar     |1280.1172340425533|47     |
|ranibizumab  |originator     |2158.908684210526 |38     |
+-------------+---------------+------------------+-----

In [0]:
# Date distribution across the quarter
from pyspark.sql import functions as F

print("Fills by month:")
(
    spark.table(RX_TABLE)
    .withColumn("month", F.month("service_date"))
    .groupBy("month")
    .agg(F.count("claim_id").alias("n_fills"))
    .orderBy("month")
    .show()
)

Fills by month:
+-----+-------+
|month|n_fills|
+-----+-------+
|   10|    198|
|   11|    180|
|   12|    174|
+-----+-------+



In [0]:
# DAW code distribution
print("DAW code distribution (pharmacy_claims):")
(
    spark.table(RX_TABLE)
    .groupBy("daw_code", "drug_type")
    .count()
    .orderBy("drug_type", "daw_code")
    .show()
)

DAW code distribution (pharmacy_claims):
+--------+---------------+-----+
|daw_code|      drug_type|count|
+--------+---------------+-----+
|       0|     biosimilar|   60|
|       7|     biosimilar|  100|
|       0|new_formulation|   25|
|       7|new_formulation|   32|
|       0|     originator|  154|
|       1|     originator|  181|
+--------+---------------+-----+



In [0]:
# Final row counts across all NB02 tables
print("=" * 60)
print("NB02 output tables - final row counts")
print("=" * 60)
for tbl in [MEMBER_TABLE, RX_TABLE, SEED_TABLE]:
    n = spark.table(tbl).count()
    print(f"  {tbl:<55}: {n:>6,}")

NB02 output tables - final row counts
  cross_benefit_audit.member_registry                    :    200
  cross_benefit_audit.pharmacy_claims                    :    552
  cross_benefit_audit.seeded_duplicate_events            :     72


## 11 - Summary

### Tables written

| Table | Rows | Key fields |
|---|---|---|
| `member_registry` | 200 | member_id, dob, plan_type, pbm_bin, primary_molecule |
| `pharmacy_claims` | 552 | claim_id, member_id, service_date, submitted_ndc, hcpcs_code, is_seeded_duplicate |
| `seeded_duplicate_events` | 72 | seed_event_id, rx_claim_id, expected_med_hcpcs, med_date_offset |

### Seeded scenario design

The `seeded_duplicate_events` table is the **event registry** NB03 reads
to generate the medical benefit leg of each seeded scenario.

| Column | Role |
|---|---|
| `seed_event_id` | Links pharmacy claim to its medical counterpart |
| `expected_med_hcpcs` | The HCPCS J/Q code NB03 should put on the medical claim |
| `med_date_offset` | Days after `rx_service_date` to date the medical claim |
| `scenario_type` | Drives NB04 detection rule logic |
| `shared_prescriber_npi` | NPI used on pharmacy prescriber_npi AND medical rendering_npi (NPI co-location signal) |
| `administered_hcpcs_units` | Units drawn from medical HCPCS range; NB03 applies ±8% noise and writes to hcpcs_units (dose concordance signal) |

### Critical detection insight - the QCODE_GAP

18 of the 72 seeded scenarios involve a biosimilar Q-code at the pharmacy and an originator J-code on the medical claim. A detection engine that only matches `pharmacy_hcpcs = medical_hcpcs` will miss **100%** of these.

The molecule-level bridge (built in NB04 using the crosswalk) is required to surface them.